# DataFrame joins in spark
A spark application bring in a large number of dataFrames
- Joins are all about bringing two dataFrames togeather and these two dataFrames are termed as left dataFrame and right dataFrame.
- We combing the left dataFrame with the right dataFrame using two things.
    - Join Condition/Expression
    - Join type
- When performing joins we always start with the left dataFrame and pass in the right dataFrame as the first argument of the join method. The join method takes in two more argument join expression and the join method.
    - ```order_df.join(product_df,join_expr,join_type)```
- Inner join is the default value of the join_type argument
- How do spark handles join internally? 
    - Spark is going to take first row from your left dataFrame and evaluate the join_expression for all the rows in the right dataFrame to find a match
    - After this as the next step Spark combines these matching records from the left and the right dataFrames to create a new dataFrame that is where the join type comes into the picture.
- There is one thing that you need to make sure you avoid and that is column abiguity 
    - You can avoid column ambiguity by re-naming those columns that are same in both the left and right dataFrames. even before performing the join operations on them.
    - Example ; 
    main.py file
    ```python
    product_renamed_df = generated_product_df.withColumnRenamed("qty","reorder_qty").withColumnRenamed("prod_id","prod_id2")
    # performing inner join opertion on the two dataFrames
    join_expression = generated_order_df.prod_id == product_renamed_df.prod_id2
    inner_join_df = df_joins.inner_join_df(left_df=generated_order_df,right_df=product_renamed_df,join_expression=join_expression)
    inner_join_df = inner_join_df.select("order_id","prod_id","unit_price","qty","prod_name","list_price","reorder_qty")
    # logging dataFrame
    sp_df_logger.log_df(spark_df=inner_join_df,spark_df_name="inner_join_df")
    ```
    joins.py file
    ```python
    from lib.logger import Log4j,LogSparkDataframe
    from lib.app_monitor import GetDataFrameMemory
    class DataFrameJoins:
        def __init__(self,spark):
            self.spark = spark
            self.logger = Log4j(spark)
            self.mem = GetDataFrameMemory(spark)
            self.metrics = LogSparkDataframe(spark)
        def inner_join_df(self,left_df,right_df,join_expression):
            try:
                # perform join operation
                result_df = left_df.join(right_df,join_expression,"inner")
                # logging memory taken by the df
                self.mem.get_mem_usage(result_df)
                # logging spark schema 
                self.logger.debug("logging intermediate dataframe after inner join operation")
                self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
                return result_df
            except Exception as e:
                self.logger.error(str(e))
                raise
        # left join is also called left outer join
        def left_join_df(self,left_df,right_df,join_expression):
            try:
                # perform join operation
                result_df = left_df.join(right_df,join_expression,"left")
                # logging memory taken by the df
                self.mem.get_mem_usage(result_df)
                # logging spark schema 
                self.logger.debug("logging intermediate dataframe after left join operation")
                self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
                return result_df
            except Exception as e:
                self.logger.error(str(e))
                raise
        # right join is also called right outer join
        def right_join_df(self,left_df,right_df,join_expression):
            try:
                # perform join operation
                result_df = left_df.join(right_df,join_expression,"right")
                # logging memory taken by the df
                self.mem.get_mem_usage(result_df)
                # logging spark schema 
                self.logger.debug("logging intermediate dataframe after right join operation")
                self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
                return result_df
            except Exception as e:
                self.logger.error(str(e))
                raise
        # here outer join is actually full outer join there is no difference between them
        def outer_join_df(self,left_df,right_df,join_expression):
            try:
                # perform join operation
                result_df = left_df.join(right_df,join_expression,"outer")
                # logging memory taken by the df
                self.mem.get_mem_usage(result_df)
                # logging spark schema 
                self.logger.debug("logging intermediate dataframe after outer join operation")
                self.metrics.log_df_metrics(spark_df=result_df,spark_df_name="result_df")
                return result_df
            except Exception as e:
                self.logger.error(str(e))
                raise
    ```



